<a href="https://colab.research.google.com/github/ArthurrCr/cloudband/blob/main/notebooks/00_baselines/score_ocm_cloudsen12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --quiet "omnicloudmask[legacy]==1.7.0" tacoreader


In [2]:
REPO_URL = "https://github.com/ArthurrCr/cloudband.git"
PROJECT_DIR = "/content/cloudband"
BRANCH = "main"

import os
import sys

if not os.path.exists(PROJECT_DIR):
    !git clone --quiet {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git fetch --quiet origin && git reset --quiet --hard origin/{BRANCH}

SRC_DIR = f"{PROJECT_DIR}/src"
os.chdir(PROJECT_DIR)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

!PYTHONPATH={SRC_DIR} python -m pytest tests -q

........................................................................ [ 74%]
.........................                                                [100%]
=============================== warnings summary ===============================
tests/contract/test_pipeline.py::test_wrong_raster_size_is_rejected
tests/contract/test_pipeline.py::test_duplicate_predictions_are_rejected
tests/contract/test_pipeline.py::test_attach_and_score_round_trip
tests/contract/test_pipeline.py::test_pooled_counts_equal_whole_collection
  /usr/local/lib/python3.13/dist-packages/rasterio/__init__.py:377: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
    dataset = writer(

tests/contract/test_pipeline.py::test_wrong_raster_size_is_rejected
tests/contract/test_pipeline.py::test_attach_and_score_round_trip
tests/contract/test_pipeline.py::test_pooled_counts_equal_whole_collection
  /usr/local/lib/python3.13/dist-packages/rasterio/__init__.py:367: No

In [ ]:
from pathlib import Path

import pandas as pd

from cloudband.baselines import ocm
from cloudband.colab.session import reload_package, start
from cloudband.datasets import cloudsen12 as ds
from cloudband.eval.palette import assign_colors
from cloudband.eval.plots import plot_experiment_comparison
from cloudband.eval.report import compare
from cloudband.pipelines import phase0

reload_package("cloudband")


In [4]:
def report(position, total):
    if position % 25 == 0 or position == total:
        print(f"{position}/{total}", flush=True)

In [5]:
from google.colab import drive

drive.mount("/content/drive")

RESULTS_DIR = Path("/content/drive/MyDrive/cloudband/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

session = start(PROJECT_DIR, require_accelerator=True)
print(f"results: {RESULTS_DIR}")

Mounted at /content/drive
project: /content/cloudband
device: cuda:Tesla T4
free disk: 65.4 GiB
omnicloudmask: 1.7.0
rasterio: 1.5.1
pandas: 2.2.3
numpy: 2.1.3
results: /content/drive/MyDrive/cloudband/results


In [6]:
table = phase0.load_test_split()
print(f"scenes: {len(table)}")
print("pairable:", ds.expected_pairable_scenes(table))

scenes: 975
pairable: {'clear': 705, 'cloud': 767, 'shadow': 655}


In [ ]:
ocm.check_version()

MODEL_VERSIONS = (ocm.PAPER_MODEL_VERSION, 2.0, 3.0, ocm.LATEST_MODEL_VERSION)
print("package:", ocm.package_version())
print("scoring model versions:", MODEL_VERSIONS)


In [ ]:
LIMIT = None

results = {}
for model_version in MODEL_VERSIONS:
    config = ocm.InferenceConfig(
        patch_size=509, patch_overlap=0, model_version=model_version
    )
    model_id = f"ocm-rgn-published-v{ocm.package_version()}-model{model_version}"
    print(f"model_version={model_version} -> model_id={model_id}")

    results[model_version] = phase0.run(
        table,
        lambda stack, config=config: ocm.predict_array(stack, config),
        model_id=model_id,
        limit=LIMIT,
        progress=report,
    )

pd.concat(
    {version: result.scores for version, result in results.items()},
    names=["model_version", "experiment"],
)


In [ ]:
comparisons = {}
for model_version, result in results.items():
    frame = phase0.compare_to_reference(result, phase0.STUPMASK_CLOUDSEN12)
    comparisons[model_version] = frame
    print(f"model_version={model_version}")
    print("  pairable scenes:", result.pairable)
    print(" ", phase0.describe_comparison(frame))

pd.concat(comparisons, names=["model_version", "experiment"])


In [ ]:
for model_version, result in results.items():
    config = ocm.InferenceConfig(
        patch_size=509, patch_overlap=0, model_version=model_version
    )
    paths = phase0.save(
        result,
        RESULTS_DIR,
        config=config.as_kwargs(),
        package_versions=session.package_versions,
    )
    print(f"model_version={model_version}")
    for name, path in paths.items():
        print(f"  {name}: {path}")


In [ ]:
def version_label(model_version: float) -> str:
    return f"V{int(model_version)}"


model_scores = {version_label(v): result.scores for v, result in results.items()}
comparison = compare(model_scores, column="boa")
colors = assign_colors(list(comparison.columns))
comparison


In [ ]:
figures = plot_experiment_comparison(comparison, colors, metric_label="BOA (%)")
for experiment, figure in figures.items():
    figure.savefig(RESULTS_DIR / f"ocm_versions_{experiment}_boa.png", dpi=150, bbox_inches="tight")
    display(figure)
